## Proteomics Quantification

Author: Troy Brier, Enguang Fu

Convert the IBAQ (Intensity-based Absolute Quantification) to "true" copy numbers in single Syn1 and Syn3A cells.

- Proteomics use IBAQ (Intensity-based Absolute Quantification)  
  IBAQ as proxy of protein copy number;  
  Yet not necessarily scale w. true protein copy number linearly;


- Input:   
  IBAQ from **Spectronaut** software

- Steps:
1. Load proteomics iBAQ from Excel
2. Convert iBAQ → iPM (iBAQ Per Million) per replicate
3. Calculate mean iPM across replicates

**Load Syn1 genome**

In [1]:
import pandas as pd
import numpy as np
import os
import pysam
from typing import Dict, List, Tuple, Optional

In [2]:
GENES_GFF    = "../Genomes_Input/syn1.genes.gff3"

# Gene annotation parsing
PRIMARY_FEATURES = {"gene"}
INCLUDE_CDS_FALLBACK = False

def parse_gff_attributes(attr: str) -> Dict[str, str]:
    out: Dict[str, str] = {}
    for item in attr.split(";"):
        item = item.strip()
        if not item:
            continue
        if "=" in item:
            k, v = item.split("=", 1)
            out[k] = v
    return out

def read_genes_gff(path: str) -> pd.DataFrame:
    rows = []
    with open(path, "r") as f:
        for line in f:
            if not line or line.startswith("#"):
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 9:
                continue
            chrom, source, feature, start1, end1, score, strand, phase, attrs = parts
            if feature == "CDS" and not INCLUDE_CDS_FALLBACK:
                continue
            if feature not in PRIMARY_FEATURES and not (INCLUDE_CDS_FALLBACK and feature == "CDS"):
                continue
            a = parse_gff_attributes(attrs)
            locus_tag = a.get("locus_tag", "")
            gene_name = a.get("gene", "")
            rna_type = a.get("rna_type", "")
            gene_product = a.get("product", "")
            s1 = int(start1); e1 = int(end1)
            start0 = s1 - 1
            end0 = e1
            rows.append((chrom, feature, start0, end0, strand, locus_tag, gene_name, rna_type, gene_product))
    df = pd.DataFrame(rows, columns=["chrom", "feature", "start0", "end0", "strand", "locus_tag", "gene_name", "rna_type", "gene_product"])
    df = df.sort_values(["chrom", "start0", "end0"]).reset_index(drop=True)
    return df

genes = read_genes_gff(GENES_GFF)

genes.head()

,chrom,feature,start0,end0,strand,locus_tag,gene_name,rna_type,gene_product
0,CP002027.1,gene,0,1353,+,MMSYN1_0001,dnaA_1,mRNA,chromosomal replication initiator protein DnaA
1,CP002027.1,gene,1510,2638,+,MMSYN1_0002,dnaN,mRNA,DNA polymerase III_ beta subunit
2,CP002027.1,gene,2674,3217,+,MMSYN1_0003,rnmV,mRNA,ribonuclease M5
3,CP002027.1,gene,3206,4007,+,MMSYN1_0004,ksgA,mRNA,dimethyladenosine transferase
4,CP002027.1,gene,4063,5155,+,MMSYN1_0005,,mRNA,hypothetical purine NTPase


**Get Molecular Weight of Proteins**

In [3]:
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis

GB_PATH = "../Genomes_Input/syn1.gb"

# Amino-acid molecular weights (monoisotopic average, Da)
# ProteinAnalysis.molecular_weight() uses standard average masses + water for the full chain
mw_map = {}
for record in SeqIO.parse(GB_PATH, "genbank"):
    for feat in record.features:
        if feat.type != "CDS":
            continue
        qualifiers = feat.qualifiers
        locus_tag  = qualifiers.get("locus_tag", [None])[0]
        translation = qualifiers.get("translation", [None])[0]
        if locus_tag is None or translation is None:
            continue
        try:
            mw = ProteinAnalysis(translation).molecular_weight()
        except Exception:
            mw = float("nan")
        mw_map[locus_tag] = mw

genes["protein_mw_Da"] = genes["locus_tag"].map(mw_map)

print(f"Proteins with MW calculated: {genes['protein_mw_Da'].notna().sum()} / {len(genes)}")
genes[["locus_tag", "gene_name", "protein_mw_Da"]].dropna().head(10)

Proteins with MW calculated: 828 / 911


,locus_tag,gene_name,protein_mw_Da
0,MMSYN1_0001,dnaA_1,51702.7783
1,MMSYN1_0002,dnaN,42649.6889
2,MMSYN1_0003,rnmV,20634.5673
3,MMSYN1_0004,ksgA,31176.8505
4,MMSYN1_0005,,42663.1420
5,MMSYN1_0006,gyrB,71583.4377
6,MMSYN1_0007,gyrA,94518.7186
7,MMSYN1_0008,,34308.3560
8,MMSYN1_0009,,97527.9433
9,MMSYN1_0010,,59922.4356


### Relative Abundance from Mass Spec

**Read in IBAQ**

In [4]:
PROTEOME_FOLDER = "../Proteomics/Processed_proteomics_canonical/"

prot = pd.read_excel(PROTEOME_FOLDER + 'Syn1_summary.xlsx', sheet_name='Proteins_all_raw data')

# Extract locus_tag
prot['locus_tag'] = prot['PG.ProteinDescriptions'].str.extract(r'\[locus_tag=(\S+?)\]')

# Rename iBAQ columns
ibaq_cols = [c for c in prot.columns if 'IBAQ' in c]
prot = prot[['locus_tag'] + ibaq_cols].copy()
prot.columns = ['locus_tag', 'iBAQ_rep1', 'iBAQ_rep2', 'iBAQ_rep3']

# Handle semicolon-separated protein groups (take first value — they are identical)
for col in ['iBAQ_rep1', 'iBAQ_rep2', 'iBAQ_rep3']:
    prot[col] = pd.to_numeric(prot[col].astype(str).str.split(';').str[0], errors='coerce')

print(f'Proteins loaded: {len(prot)}')
print(f'iBAQ NaNs: rep1={prot["iBAQ_rep1"].isna().sum()}, rep2={prot["iBAQ_rep2"].isna().sum()}, rep3={prot["iBAQ_rep3"].isna().sum()}')

Proteins loaded: 739
iBAQ NaNs: rep1=1, rep2=5, rep3=1


**Convert iBAQ → iPM and compute mean**  

$$\text{iPM}_i = \frac{\text{iBAQ}_i}{\sum_j \text{iBAQ}_j} \times 10^6$$

In [5]:
for rep in ['rep1', 'rep2', 'rep3']:
    prot[f'iPM_{rep}'] = (prot[f'iBAQ_{rep}'] / prot[f'iBAQ_{rep}'].sum()) * 1e6

prot['iPM_mean'] = prot[['iPM_rep1', 'iPM_rep2', 'iPM_rep3']].mean(axis=1)
prot['iPM_CV'] = prot[['iPM_rep1', 'iPM_rep2', 'iPM_rep3']].std(axis=1) / prot['iPM_mean']

print(f'Proteins with valid iPM_mean: {prot["iPM_mean"].notna().sum()}')
print(f'Median CV across replicates: {prot["iPM_CV"].median():.3f}')

Proteins with valid iPM_mean: 739
Median CV across replicates: 0.081


**Map proteomics iPM onto the `genes` dataframe**

In [6]:
# Build mapping from locus_tag to iPM values
iPM_map = prot.set_index('locus_tag')[['iPM_rep1', 'iPM_rep2', 'iPM_rep3', 'iPM_mean', 'iPM_CV']]

# Map onto genes dataframe
for col in iPM_map.columns:
    genes[col] = genes['locus_tag'].map(iPM_map[col])

detected = genes['iPM_mean'].notna().sum()
total = len(genes[genes['rna_type'] == 'mRNA'])
print(f'Total genes in dataframe: {total}')
print(f'Genes with proteomics data: {detected} ({detected/total*100:.1f}%)')
print(f'Genes without proteomics:   {total - detected}')

Total genes in dataframe: 828
Genes with proteomics data: 739 (89.3%)
Genes without proteomics:   89


### Convert to absolute protein copy number

**Calculate Average Molecular Weight**

In [7]:
# Restrict to genes with both MW and iPM_mean
valid = genes[["locus_tag", "gene_name", "protein_mw_Da", "iPM_mean"]].dropna()

weights = valid["iPM_mean"].values
mws     = valid["protein_mw_Da"].values

avg_mw_weighted = np.average(mws, weights=weights)
avg_mw_unweighted = mws.mean()

print(f"Proteins used (have both MW and iPM_mean): {len(valid)}")
print(f"  iPM_mean-weighted average MW : {avg_mw_weighted:,.1f} Da  ({avg_mw_weighted/1e3:.2f} kDa)")
print(f"  Unweighted average MW        : {avg_mw_unweighted:,.1f} Da  ({avg_mw_unweighted/1e3:.2f} kDa)")

valid.sort_values("iPM_mean", ascending=False).head(10)[
    ["locus_tag", "gene_name", "protein_mw_Da", "iPM_mean"]
]

Proteins used (have both MW and iPM_mean): 739
  iPM_mean-weighted average MW : 35,489.8 Da  (35.49 kDa)
  Unweighted average MW        : 44,687.7 Da  (44.69 kDa)


,locus_tag,gene_name,protein_mw_Da,iPM_mean
155,MMSYN1_0151,tuf,43303.6945,54291.790586
473,MMSYN1_0475,ldh,34573.2154,34088.220832
227,MMSYN1_0223,,50178.5330,29846.385272
229,MMSYN1_0225,pdhA,41904.0844,29692.624458
607,MMSYN1_0607,gap,36984.9536,21891.165720
667,MMSYN1_0667,rpsS,9860.2906,18788.184503
665,MMSYN1_0665,rpsC,26215.3626,15921.758988
64,MMSYN1_0059,ald,40575.4955,15389.154193
217,MMSYN1_0213,eno,49506.9911,15363.779521
351,MMSYN1_0350,hup,10011.5381,15019.848108


In [8]:
NA           = 6.0221409e23
gDW          = 1.2844977017695941e-14  # cell dry mass (g), from calc
ptn_mass_frac = 0.582                  # avg, ranges 50.7–62.4% (Razin et al. 1963)

radius  = 439 / 2                              # nm, Syn1.0 (Roger-Reischer et al. Nature 2023)
vol_m3  = 4.0/3.0 * np.pi * radius**3 * 1e-27 # m^3
vol_um3 = 4.0/3.0 * np.pi * radius**3 * 1e-9  # um^3

# Per-rep relative abundance from iPM (iPM / 1e6)
avg_ptn_mw    = np.zeros(3)
total_n_ptns  = np.zeros(3)

valid = genes[["protein_mw_Da", "iPM_rep1", "iPM_rep2", "iPM_rep3"]].dropna()

for i, rep in enumerate(["iPM_rep1", "iPM_rep2", "iPM_rep3"]):
    rel_abund = valid[rep] / valid[rep].sum()          # relative abundance within rep
    avg_ptn_mw[i] = np.sum(valid["protein_mw_Da"] * rel_abund)
    total_n_ptns[i] = (gDW * ptn_mass_frac) / (avg_ptn_mw[i] / NA)
    print(f"Rep {i+1} avg protein MW (g/mol)      : {avg_ptn_mw[i]:,.1f}")
    print(f"Rep {i+1} total protein molecules/cell: {total_n_ptns[i]:,.0f}")
    print(f"Rep {i+1} protein density (ptns/um^3) : {total_n_ptns[i]/vol_um3:.3e}")
    print()

print(f"Mean total proteins/cell : {total_n_ptns.mean():,.0f}  ± {total_n_ptns.std():,.0f}")
print(f"Mean protein density     : {(total_n_ptns/vol_um3).mean():.3e} ptns/um^3")


Rep 1 avg protein MW (g/mol)      : 35,454.5
Rep 1 total protein molecules/cell: 126,980
Rep 1 protein density (ptns/um^3) : 2.866e+06

Rep 2 avg protein MW (g/mol)      : 35,752.5
Rep 2 total protein molecules/cell: 125,922
Rep 2 protein density (ptns/um^3) : 2.843e+06

Rep 3 avg protein MW (g/mol)      : 35,262.0
Rep 3 total protein molecules/cell: 127,673
Rep 3 protein density (ptns/um^3) : 2.882e+06

Mean total proteins/cell : 126,858  ± 720
Mean protein density     : 2.864e+06 ptns/um^3


In [9]:
# Absolute copy number per protein = (iPM_mean / 1e6) * mean total proteins/cell
genes["ptn_copy_number"] = (genes["iPM_mean"] / 1e6) * total_n_ptns.mean()

print(f"Mean total proteins/cell used: {total_n_ptns.mean():,.0f}")
print(f"Genes with copy number assigned: {genes['ptn_copy_number'].notna().sum()}")
print()
genes[["locus_tag", "gene_name", "gene_product", "iPM_mean", "ptn_copy_number"]].dropna(
    subset=["ptn_copy_number"]
).sort_values("ptn_copy_number", ascending=False).head(10)

Mean total proteins/cell used: 126,858
Genes with copy number assigned: 739



,locus_tag,gene_name,gene_product,iPM_mean,ptn_copy_number
155,MMSYN1_0151,tuf,translation elongation factor Tu,54291.790586,6887.364023
473,MMSYN1_0475,ldh,L-lactate dehydrogenase,34088.220832,4324.373597
227,MMSYN1_0223,,NADH oxidase (noxase),29846.385272,3786.261568
229,MMSYN1_0225,pdhA,pyruvate dehydrogenase (acetyl-transferring) E...,29692.624458,3766.755733
607,MMSYN1_0607,gap,glyceraldehyde-3-phosphate dehydrogenase_ type I,21891.165720,2777.075974
667,MMSYN1_0667,rpsS,ribosomal protein S19,18788.184503,2383.437065
665,MMSYN1_0665,rpsC,ribosomal protein S3,15921.758988,2019.807209
64,MMSYN1_0059,ald,alanine dehydrogenase,15389.154193,1952.241873
217,MMSYN1_0213,eno,phosphopyruvate hydratase,15363.779521,1949.022885
351,MMSYN1_0350,hup,DNA-binding protein HU,15019.848108,1905.392332


## Proteome Localizaiton

Localize protein in Syn1 as did in Syn3A in paper [Assembly of Macromolecular Complexes in the Whole-cell Model of a Minimal Cell](https://doi.org/10.1021/acs.jpcb.5c04532)

- Purpose:  
Exclude membrane proteins to check the better correlation between transcriptomics and proteomics.

- Input:  
  Topology predictions of membrane proteins from DeepTMHMM   
  Signal peptides predictions from SingalP 6  
  Proteomics of Syn1 and Syn3A

**Parse DeepTMHMM**

In [10]:
import re

DEEPTMHMM_GFF3 = "./DeepTMHMM_Topology/TMRs_syn1.gff3"

# Extract locus_tag robustly — handles both separator styles:
#   "MMSYN1_0001_gene_dnaA_1_product_..."  and  "MMSYN1_0269|gene=|product=..."
LOCUS_RE = re.compile(r"(MMSYN1_\d+)")

topo_dict = {}   # keyed by locus_tag e.g. "MMSYN1_0005"

with open(DEEPTMHMM_GFF3) as fh:
    current_tag = None
    for line in fh:
        line = line.rstrip("\n")
        if not line or line.startswith("##") or line == "//":
            continue
        if line.startswith("#"):
            body = line[2:].strip()
            m = LOCUS_RE.search(body)
            if not m:
                continue
            current_tag = m.group(1)
            topo_dict.setdefault(current_tag, {"TMRs": 0, "regions": [], "length": None})
            if "Length:" in body:
                topo_dict[current_tag]["length"] = int(re.search(r"Length:\s*(\d+)", body).group(1))
            elif "Number of predicted TMRs:" in body:
                topo_dict[current_tag]["TMRs"]   = int(re.search(r"TMRs:\s*(\d+)", body).group(1))
        else:
            # data line: seq_id <tab> region <tab> start <tab> end ...
            parts = line.split("\t")
            if len(parts) < 4 or current_tag is None:
                continue
            region = parts[1].strip()
            start  = int(parts[2].strip())
            end    = int(parts[3].strip())
            topo_dict[current_tag]["regions"].append((region, start, end))

print(f"Proteins parsed from DeepTMHMM: {len(topo_dict)}")
print(f"  Transmembrane (TMRs > 0): {sum(1 for v in topo_dict.values() if v['TMRs'] > 0)}")
print(f"  Cytoplasmic and Others   (TMRs = 0): {sum(1 for v in topo_dict.values() if v['TMRs'] == 0)}")

# Sanity check
for tag in list(topo_dict)[:3]:
    print(f"  {tag}: TMRs={topo_dict[tag]['TMRs']}, length={topo_dict[tag]['length']}, regions={topo_dict[tag]['regions'][:2]}")

Proteins parsed from DeepTMHMM: 828
  Transmembrane (TMRs > 0): 162
  Cytoplasmic and Others   (TMRs = 0): 666
  MMSYN1_0001: TMRs=0, length=450, regions=[('inside', 1, 450)]
  MMSYN1_0002: TMRs=0, length=375, regions=[('inside', 1, 375)]
  MMSYN1_0003: TMRs=0, length=180, regions=[('inside', 1, 180)]


In [11]:
topo_dict

{'MMSYN1_0001': {'TMRs': 0, 'regions': [('inside', 1, 450)], 'length': 450},
 'MMSYN1_0002': {'TMRs': 0, 'regions': [('inside', 1, 375)], 'length': 375},
 'MMSYN1_0003': {'TMRs': 0, 'regions': [('inside', 1, 180)], 'length': 180},
 'MMSYN1_0004': {'TMRs': 0, 'regions': [('inside', 1, 266)], 'length': 266},
 'MMSYN1_0005': {'TMRs': 4,
  'regions': [('inside', 1, 38),
   ('TMhelix', 39, 59),
   ('outside', 60, 296),
   ('TMhelix', 297, 314),
   ('inside', 315, 326),
   ('TMhelix', 327, 336),
   ('outside', 337, 337),
   ('TMhelix', 338, 348),
   ('inside', 349, 363)],
  'length': 363},
 'MMSYN1_0006': {'TMRs': 0, 'regions': [('inside', 1, 634)], 'length': 634},
 'MMSYN1_0007': {'TMRs': 0, 'regions': [('inside', 1, 834)], 'length': 834},
 'MMSYN1_0008': {'TMRs': 9,
  'regions': [('outside', 1, 12),
   ('TMhelix', 13, 28),
   ('inside', 29, 37),
   ('TMhelix', 38, 55),
   ('outside', 56, 70),
   ('TMhelix', 71, 86),
   ('inside', 87, 103),
   ('TMhelix', 104, 113),
   ('outside', 114, 157)

**SignalP 6**

In [12]:
SIGNALP_GFF3 = "./SignalP_prediction/output.gff3"

# Parse SignalP 6 gff3 for Syn1
# Data lines: "MMSYN1_XXXX_gene_...<tab>SignalP-6.0<tab>signal_peptide|lipoprotein_signal_peptide<tab>..."
signalP = {}   # keyed by locus_tag

with open(SIGNALP_GFF3) as fh:
    for line in fh:
        line = line.rstrip("\n")
        if not line or line.startswith("#"):
            continue
        parts = line.split("\t")
        if len(parts) < 3:
            continue
        seq_id     = parts[0].strip()           # e.g. MMSYN1_0011_gene__product_...
        prediction = parts[2].strip()
        locus_tag  = seq_id.split("_gene_")[0]  # e.g. MMSYN1_0011
        if prediction == "signal_peptide":
            signalP[locus_tag] = "extracellular"
        elif prediction == "lipoprotein_signal_peptide":
            signalP[locus_tag] = "lipoprotein"

print(f"Proteins with SignalP prediction: {len(signalP)}")
print(f"  extracellular : {sum(1 for v in signalP.values() if v == 'extracellular')}")
print(f"  lipoprotein   : {sum(1 for v in signalP.values() if v == 'lipoprotein')}")

Proteins with SignalP prediction: 93
  extracellular : 15
  lipoprotein   : 78


**Output proteomics_topo**

In [13]:
# ── Map DeepTMHMM predictions onto genes ─────────────────────────────────────
genes["TMRs"]          = genes["locus_tag"].map(lambda t: topo_dict.get(t, {}).get("TMRs",      None))
genes["topo_regions"]  = genes["locus_tag"].map(lambda t: topo_dict.get(t, {}).get("regions",   None))
genes["protein_length"]= genes["locus_tag"].map(lambda t: topo_dict.get(t, {}).get("length",    None))

# ── Map SignalP predictions onto genes ────────────────────────────────────────
genes["signalP"] = genes["locus_tag"].map(signalP)   # extracellular | lipoprotein | NaN

# ── Assign localization ───────────────────────────────────────────────────────
# Priority: SignalP (secreted/lipoprotein) > DeepTMHMM (membrane) > cytoplasmic
def assign_localization(row):
    if row['rna_type'] != 'mRNA':
        return "Non_proteins"                        # non-coding RNA, no prediction available
    if pd.notna(row["signalP"]):
        return row["signalP"]                          # extracellular or lipoprotein
    tmrs = row["TMRs"]
    if pd.notna(tmrs) and tmrs > 0:
        return "membrane"
    if pd.notna(tmrs) and tmrs == 0:
        return "cytoplasmic"
    return "Non_predicted"                                        # no prediction available

genes["ptn_localization"] = genes.apply(assign_localization, axis=1)

# ── Summary ───────────────────────────────────────────────────────────────────
print("Localization summary:")
print(f"Total proteins (mRNA): {genes[genes['rna_type'] == 'mRNA'].shape[0]}")
print(genes[genes['rna_type'] == 'mRNA']['ptn_localization'].value_counts(dropna=False).to_string())
print()
genes[["locus_tag", "gene_name", "TMRs", "signalP", "ptn_localization"]].head(15)

Localization summary:
Total proteins (mRNA): 828
ptn_localization
cytoplasmic      582
membrane         153
lipoprotein       78
extracellular     15



,locus_tag,gene_name,TMRs,signalP,ptn_localization
0,MMSYN1_0001,dnaA_1,0.0,NaN,cytoplasmic
1,MMSYN1_0002,dnaN,0.0,NaN,cytoplasmic
2,MMSYN1_0003,rnmV,0.0,NaN,cytoplasmic
3,MMSYN1_0004,ksgA,0.0,NaN,cytoplasmic
4,MMSYN1_0005,,4.0,NaN,membrane
5,MMSYN1_0006,gyrB,0.0,NaN,cytoplasmic
6,MMSYN1_0007,gyrA,0.0,NaN,cytoplasmic
7,MMSYN1_0008,,9.0,NaN,membrane
8,MMSYN1_0009,,10.0,NaN,membrane
9,MMSYN1_0010,,0.0,NaN,cytoplasmic


In [14]:
genes.to_csv("./syn1_proteomics_localization_2026.csv", index=False)

## Syn3A Proteomics Quantification

**Load Syn3A reference proteome and map molecular weight from Syn1**

In [15]:
SYN3A_CPLX_XLSX  = "./syn3A_updated_proteome_cplx.xlsx" # From complex formation paper

# Load Syn3A reference proteome (skip row 0 which is a header note)
syn3a = pd.read_excel(SYN3A_CPLX_XLSX, sheet_name="Comparative Proteomics", skiprows=[1])
selected_cols = ["Locus Tag", "Gene Name", "Gene Product", "Essentiality", "Protein Length", "Exp. Ptn Cnt", "Localization"]
syn3a = syn3a[selected_cols]
syn3a = syn3a.rename(columns={"Locus Tag": "locus_tag", "Gene Name": "gene_name",
                               "Gene Product": "gene_product", "Essentiality": "essentiality", "Protein Length": "protein_length",
                               "Exp. Ptn Cnt": "exp_ptn_cnt_2019", "Localization": "localization"})
syn3a = syn3a.dropna(subset=["locus_tag"]).reset_index(drop=True)

# Extract locus number (e.g. "JCVISYN3A_0001" -> "0001") to match Syn1 MW by gene order
# Syn1 and Syn3A share the same gene numbering
syn3a["locus_num"] = syn3a["locus_tag"].str.extract(r"_(\d+)$")
genes["locus_num"]  = genes["locus_tag"].str.extract(r"_(\d+)$")

mw_by_num = genes.dropna(subset=["protein_mw_Da"]).set_index("locus_num")["protein_mw_Da"]
syn3a["protein_mw_Da"] = syn3a["locus_num"].map(mw_by_num)

print(f"Syn3A proteins loaded       : {len(syn3a)}")
print(f"With MW from Syn1 genes df  : {syn3a['protein_mw_Da'].notna().sum()}")
syn3a[["locus_tag", "gene_name", "essentiality", "protein_length", "protein_mw_Da", "exp_ptn_cnt_2019", "localization"]].head(8)

syn3a = syn3a.drop(columns=["locus_num"])  # cleanup

Syn3A proteins loaded       : 455
With MW from Syn1 genes df  : 445


In [23]:
syn3a[syn3a["protein_mw_Da"] == ""]

,locus_tag,gene_name,gene_product,essentiality,protein_length,exp_ptn_cnt_2019,localization,protein_mw_Da,iBAQ_rep1,iBAQ_rep2,iBAQ_rep3,iPM_rep1,iPM_rep2,iPM_rep3,iPM_mean,iPM_CV,copy_number_2026


**Calculate iBAQ → iPM and mean across replicates**

In [16]:
SYN3A_SUMMARY = "./Processed_proteomics_canonical/Syn3_summary.xlsx"

syn3_raw = pd.read_excel(SYN3A_SUMMARY, sheet_name="Proteins_all_raw data")

# Extract locus_tag and IBAQ columns
syn3_raw["locus_tag"] = syn3_raw["PG.ProteinDescriptions"].str.extract(r"\[locus_tag=(\S+?)\]")
ibaq_cols = [c for c in syn3_raw.columns if "IBAQ" in c]
syn3_prot = syn3_raw[["locus_tag"] + ibaq_cols].copy()
syn3_prot.columns = ["locus_tag", "iBAQ_rep1", "iBAQ_rep2", "iBAQ_rep3"]

for col in ["iBAQ_rep1", "iBAQ_rep2", "iBAQ_rep3"]:
    syn3_prot[col] = pd.to_numeric(
        syn3_prot[col].astype(str).str.split(";").str[0], errors="coerce"
    )

# iPM per replicate
for rep in ["rep1", "rep2", "rep3"]:
    syn3_prot[f"iPM_{rep}"] = syn3_prot[f"iBAQ_{rep}"] / syn3_prot[f"iBAQ_{rep}"].sum() * 1e6

syn3_prot["iPM_mean"] = syn3_prot[["iPM_rep1", "iPM_rep2", "iPM_rep3"]].mean(axis=1)
syn3_prot["iPM_CV"]   = (syn3_prot[["iPM_rep1", "iPM_rep2", "iPM_rep3"]].std(axis=1)
                         / syn3_prot["iPM_mean"])

print(f"Syn3A proteins with iBAQ data: {len(syn3_prot)}")
print(f"iBAQ NaNs: rep1={syn3_prot['iBAQ_rep1'].isna().sum()}, "
      f"rep2={syn3_prot['iBAQ_rep2'].isna().sum()}, rep3={syn3_prot['iBAQ_rep3'].isna().sum()}")
print(f"Median CV: {syn3_prot['iPM_CV'].median():.3f}")

# Map onto syn3a dataframe
iPM_map = syn3_prot.set_index("locus_tag")[["iBAQ_rep1","iBAQ_rep2","iBAQ_rep3",
                                             "iPM_rep1","iPM_rep2","iPM_rep3",
                                             "iPM_mean","iPM_CV"]]
for col in iPM_map.columns:
    syn3a[col] = syn3a["locus_tag"].map(iPM_map[col])

# print(f"\nSyn3A genes with iPM_mean: {syn3a['iPM_mean'].notna().sum()} / {len(syn3a)}")
print(f"Percentage of protein detected out of {len(syn3a)}: {syn3a['iPM_mean'].notna().sum() / len(syn3a) * 100:.1f}%")
syn3a[["locus_tag", "gene_name", "iPM_rep1", "iPM_rep2", "iPM_rep3", "iPM_mean"]].dropna().head(8)

Syn3A proteins with iBAQ data: 449
iBAQ NaNs: rep1=1, rep2=0, rep3=0
Median CV: 0.046
Percentage of protein detected out of 455: 98.0%


,locus_tag,gene_name,iPM_rep1,iPM_rep2,iPM_rep3,iPM_mean
0,JCVISYN3A_0001,dnaA,710.766992,730.460186,788.955121,743.394100
1,JCVISYN3A_0002,dnaN,2772.675140,2633.602614,2639.345802,2681.874519
2,JCVISYN3A_0003,rnmV,487.978623,495.591827,473.426754,485.665735
3,JCVISYN3A_0004,ksgA,169.309346,176.035188,163.561354,169.635296
5,JCVISYN3A_0006,gyrB,1006.534166,968.615648,936.088877,970.412897
6,JCVISYN3A_0007,gyrA,1772.915521,1768.006506,1698.281900,1746.401309
7,JCVISYN3A_0008,rnsD; uptD,130.033027,110.771481,123.512274,121.438927
8,JCVISYN3A_0009,rnsC; uptD,247.790547,247.637433,235.583773,243.670584


**Calculate average MW and absolute protein copy number per cell**

In [17]:
# ── Syn3A physical constants ──────────────────────────────────────────────────
# Syn3A dry mass: Breuer et al. eLife 2019 / Luthey-Schulten et al. 2022
NA            = 6.0221409e23
gDW_syn3a = 1.0161e-14 #g #from Breuer et al eLife 2019
ptn_mass_frac = 54.727/100 #from Breuer et al eLife 2019
# gDW_syn3a     = 5.4e-15           # g, Syn3A dry mass (Breuer et al. eLife 2019)
# ptn_mass_frac = 0.582             # same protein fraction used for Syn1
radius_syn3a  = 200               # nm, Syn3A radius (spherical, ~0.034 um^3 volume)
vol_um3_syn3a = 4.0/3.0 * np.pi * (radius_syn3a)**3 * 1e-9  # um^3

# ── Per-rep weighted average MW and total protein count ───────────────────────
valid3a = syn3a[["protein_mw_Da", "iPM_rep1", "iPM_rep2", "iPM_rep3"]].dropna()

avg_ptn_mw_3a   = np.zeros(3)
total_n_ptns_3a = np.zeros(3)

for i, rep in enumerate(["iPM_rep1", "iPM_rep2", "iPM_rep3"]):
    rel_abund = valid3a[rep] / valid3a[rep].sum()
    avg_ptn_mw_3a[i]   = np.sum(valid3a["protein_mw_Da"] * rel_abund)
    total_n_ptns_3a[i] = (gDW_syn3a * ptn_mass_frac) / (avg_ptn_mw_3a[i] / NA)
    print(f"Rep {i+1} avg protein MW (g/mol)      : {avg_ptn_mw_3a[i]:,.1f}")
    print(f"Rep {i+1} total protein molecules/cell: {total_n_ptns_3a[i]:,.0f}")
    print(f"Rep {i+1} protein density (ptns/um^3) : {total_n_ptns_3a[i]/vol_um3_syn3a:.3e}")
    print()

print(f"Mean total proteins/cell : {total_n_ptns_3a.mean():,.0f}  ± {total_n_ptns_3a.std():,.0f}")
print(f"Mean protein density     : {(total_n_ptns_3a/vol_um3_syn3a).mean():.3e} ptns/um^3")

# ── Absolute copy number per protein ─────────────────────────────────────────
syn3a["copy_number_2026"] = (syn3a["iPM_mean"] / 1e6) * total_n_ptns_3a.mean()

print()
print("Top 10 most abundant Syn3A proteins:")
syn3a[["locus_tag", "gene_name", "gene_product", "iPM_mean", "copy_number_2026", "exp_ptn_cnt_2019"]].dropna(
    subset=["copy_number_2026"]
).sort_values("copy_number_2026", ascending=False).head(10)

Rep 1 avg protein MW (g/mol)      : 33,433.2
Rep 1 total protein molecules/cell: 100,164
Rep 1 protein density (ptns/um^3) : 2.989e+06

Rep 2 avg protein MW (g/mol)      : 33,457.5
Rep 2 total protein molecules/cell: 100,091
Rep 2 protein density (ptns/um^3) : 2.987e+06

Rep 3 avg protein MW (g/mol)      : 33,213.1
Rep 3 total protein molecules/cell: 100,828
Rep 3 protein density (ptns/um^3) : 3.009e+06

Mean total proteins/cell : 100,361  ± 331
Mean protein density     : 2.995e+06 ptns/um^3

Top 10 most abundant Syn3A proteins:


,locus_tag,gene_name,gene_product,iPM_mean,copy_number_2026,exp_ptn_cnt_2019
76,JCVISYN3A_0151,tuf,Translation elongation factor Tu,50711.589441,5089.462038,542
243,JCVISYN3A_0475,ldh,L-lactate dehydrogenase,41112.017966,4126.040162,1100
345,JCVISYN3A_0665,rpsC,30S ribosomal protein S3,28467.638707,2857.038561,1001
327,JCVISYN3A_0647,rpsM,30S ribosomal protein S13,26477.146761,2657.270947,665
300,JCVISYN3A_0607,gapDH,Type I glyceraldehyde-3-phosphate dehydrogenase,22688.155705,2277.004299,1355
347,JCVISYN3A_0667,rpsS,30S ribosomal protein S19,19885.890226,1995.766343,630
404,JCVISYN3A_0806,rplG,50S ribosomal protein L7/L12,19779.677549,1985.106740,765
339,JCVISYN3A_0659,rplE,50S ribosomal protein L5,17006.897708,1706.827990,703
74,JCVISYN3A_0149,rpsG,30S ribosomal protein S7,15666.069493,1572.261230,498
338,JCVISYN3A_0658,rpsN,30S ribosomal protein S14,15370.699878,1542.617662,331


In [18]:
syn3a

,locus_tag,gene_name,gene_product,essentiality,protein_length,exp_ptn_cnt_2019,localization,protein_mw_Da,iBAQ_rep1,iBAQ_rep2,iBAQ_rep3,iPM_rep1,iPM_rep2,iPM_rep3,iPM_mean,iPM_CV,copy_number_2026
0,JCVISYN3A_0001,dnaA,Chromosomal replication initiator protein,Essential,450,148,cytoplasm,51702.7783,7277890.0,7414971.9,8452846.6,710.766992,730.460186,788.955121,743.394100,0.054705,74.607720
1,JCVISYN3A_0002,dnaN,DNA polymerase III subunit beta,Essential,375,213,cytoplasm,42649.6889,28390773.5,26733954.5,28277888.8,2772.675140,2633.602614,2639.345802,2681.874519,0.029341,269.155408
2,JCVISYN3A_0003,rnmV,Ribonuclease M5,Quasiessential,180,65,cytoplasm,20634.5673,4996651.2,5030800.5,5072283.1,487.978623,495.591827,473.426754,485.665735,0.023189,48.741863
3,JCVISYN3A_0004,ksgA,16S rRNA (adenine(1518)-N(6)/adenine(1519)-N(6...,Nonessential,266,40,cytoplasm,31176.8505,1733641.0,1786950.2,1752392.5,169.309346,176.035188,163.561354,169.635296,0.036804,17.024755
4,JCVISYN3A_0005,NaN,Uncharacterized protein,Nonessential,363,72,trans-membrane,42663.1420,3109443.6,3133194.2,3181872.8,303.671789,308.655737,296.983367,303.103631,0.019323,30.419761
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
450,JCVISYN3A_0906,NaN,Uncharacterized protein,Essential,238,75,cytoplasm,28274.0150,5132781.8,6142196.4,5725330.1,501.273292,605.077131,534.379567,546.909997,0.096952,54.888393
451,JCVISYN3A_0907,NaN,Low specificity hydrolase,Quasiessential,264,30,cytoplasm,31202.1823,1722936.4,1769012.1,1499401.6,168.263923,174.268079,139.948189,160.826730,0.113967,16.140719
452,JCVISYN3A_0908,yidC,Membrane protein insertase,Essential,396,120,trans-membrane,45467.4300,4094287.1,3760728.6,4212177.7,399.852722,370.475108,393.147933,387.825254,0.039696,38.922501
453,JCVISYN3A_0909,rnpA,Ribonuclease P protein component,Essential,109,6,cytoplasm,12871.3195,293731.3,274385.7,263914.7,28.686132,27.030154,24.632750,26.783012,0.076092,2.687968


In [19]:
syn3a.to_csv("./syn3a_proteomics_summary_2026.csv", index=False)